# 05 — 128x128 patch extraction (S1, S2)

`TEST_MODE=True` first, check the audit, then `False`. Per source: `s2`, then `s1` descending, then `s1` ascending.

Output per tile: `tile_<tx>_<ty>.zarr.zip` `(point, time, band, 128, 128)` uint16 (chunked one-patch-per-chunk), `.meta.json` (tile-level fields + per-sample `samples` list), `.points.parquet`. Decode: `value * scale_factor` where `value != 0`.

Built on the Colab VM's local disk, shipped to Drive as one zip per tile (writing ~21,000 chunk files straight to Drive is unusably slow).

## Setup

In [ ]:
!pip -q install odc-stac pystac-client planetary-computer rioxarray "zarr<3" geopandas pyarrow dask

In [ ]:
import os, json, glob, time, shutil, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import zarr
assert zarr.__version__.startswith('2.'), (
    f"zarr {zarr.__version__} installed -- this notebook needs zarr 2.x "
    "(zarr.Blosc / zarr.open(..., compressor=...) are v2 APIs, removed in v3). "
    "Restart the runtime after the pip cell above so the pin actually takes effect: "
    "Runtime -> Restart session, then re-run from the top WITHOUT re-running pip "
    "(or re-run pip then restart again).")
print("zarr version OK:", zarr.__version__)
import planetary_computer as pc
from pystac_client import Client
from odc.stac import load as odc_load

warnings.filterwarnings('ignore')
os.environ['GDAL_HTTP_MAX_RETRY']    = '5'
os.environ['GDAL_HTTP_RETRY_DELAY']  = '2'
os.environ['CPL_VSIL_CURL_USE_HEAD'] = 'NO'

from google.colab import drive
drive.mount('/content/drive')

ROOT        = '/content/drive/MyDrive/Crop_Classification'
DIR_POINTS  = f'{ROOT}/01_Points'
DIR_PATCHES = f'{ROOT}/04_Patches'
DIR_QA      = f'{ROOT}/05_QA'
os.makedirs(DIR_PATCHES, exist_ok=True); os.makedirs(DIR_QA, exist_ok=True)

# Local scratch on the Colab VM. Each tile's zarr is built HERE (thousands of small
# chunk files), then zipped and the single .zip is moved to Drive.
#
# Why: a one-patch-per-chunk zarr writes ~21,000 files per tile. Writing those straight
# to a mounted Drive is unusably slow and produces millions of files. One zip per tile
# gives Drive ~3 files per tile instead, and zarr reads a zip directly via ZipStore.
WORK_DIR = '/content/work'
os.makedirs(WORK_DIR, exist_ok=True)

POINTS_GPKG  = f'{DIR_POINTS}/wbcrop_points_extended.gpkg'
POINTS_LAYER = 'wbcrop_points_extended'

free_gb = shutil.disk_usage('/content').free / 1e9
print("points   :", POINTS_GPKG)
print("patches  :", DIR_PATCHES)
print("scratch  :", WORK_DIR, f"| local free: {free_gb:,.0f} GB")
print("\nTiles are staged locally then zipped to Drive — do not point WORK_DIR at Drive.")

## Configuration

`TEST_MODE` is the only switch between trial and full run.

In [ ]:
TEST_MODE     = True       # True = small sample + audit.  False = full state.
TEST_N_POINTS = 15         # points in the test tile
TEST_TILE_RANK = 0         # 0 = densest tile; try another if it is unrepresentative

SOURCE = 's2'              # 's2' or 's1'
ORBIT  = 'descending'      # S1 only: 'descending', then 'ascending'

START_DATE = '2023-01-01'
END_DATE   = '2025-01-01'  # exclusive -> through 31 Dec 2024

PATCH    = 128
RES_M    = 10
TILE_DEG = 0.1
HALO_DEG = 0.008           # > half a patch (640 m) so edge points get a full window

MAX_SCENE_CLOUD = 100      # S2: 100 keeps every scene (raw)
MAX_RETRIES     = 3
FORCE           = False

if SOURCE == 's2':
    COLLECTION   = 'sentinel-2-l2a'
    BANDS        = ['B02','B03','B04','B05','B06','B07','B08','B8A','B11','B12','SCL']
    SCALE_FACTOR = 1e-4
    ENCODE       = 'native_uint16'
    TAG          = 's2'
else:
    COLLECTION   = 'sentinel-1-rtc'
    BANDS        = ['vv','vh']
    SCALE_FACTOR = 1e-4
    ENCODE       = 'linear_x10000'
    TAG          = f"s1_{'asc' if ORBIT.startswith('asc') else 'desc'}"

NODATA = 0
RUN_ID = f"{TAG}_{PATCH}px_{pd.Timestamp(START_DATE):%Y%m}_{pd.Timestamp(END_DATE):%Y%m}"
if TEST_MODE:
    RUN_ID += "_TEST"
STORE_DIR = os.path.join(DIR_PATCHES, RUN_ID)
os.makedirs(STORE_DIR, exist_ok=True)

catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1",
                      modifier=pc.sign_inplace)

print("MODE   :", "TEST" if TEST_MODE else "FULL RUN")
print("SOURCE :", SOURCE, "|", COLLECTION, "| bands:", len(BANDS))
print("ENCODE :", ENCODE, "| scale_factor:", SCALE_FACTOR, "| nodata:", NODATA)
print("STORE  :", STORE_DIR)
print(f"one patch = {PATCH*PATCH*len(BANDS)*2/1e6:.2f} MB uncompressed")

## Points

In [ ]:
pts = gpd.read_file(POINTS_GPKG, layer=POINTS_LAYER).to_crs('EPSG:4326')
assert pts['id'].is_unique, "duplicate ids in the points file"

pts['tx'] = np.floor(pts.geometry.x / TILE_DEG).astype(int)
pts['ty'] = np.floor(pts.geometry.y / TILE_DEG).astype(int)
tiles_all = pts.groupby(['tx','ty']).size().sort_values(ascending=False)

if TEST_MODE:
    tkey = tiles_all.index[TEST_TILE_RANK]
    pts = pts[(pts['tx'] == tkey[0]) & (pts['ty'] == tkey[1])].head(TEST_N_POINTS).copy()
    print(f"TEST: tile {tkey}, {len(pts)} points, crops: "
          f"{sorted(pts['crop'].unique())}")

tiles = pts.groupby(['tx','ty']).size().sort_values(ascending=False)
N_INPUT = len(pts)
print(f"Points: {N_INPUT:,} | tiles: {len(tiles)}")

def tile_bbox(tx, ty, buf=0.0):
    return [tx*TILE_DEG - buf, ty*TILE_DEG - buf,
            (tx+1)*TILE_DEG + buf, (ty+1)*TILE_DEG + buf]

def utm_epsg(lon, lat):
    zone = int((lon + 180) // 6) + 1
    return f"EPSG:{(32600 if lat >= 0 else 32700) + zone}"

tile_crs = {}
for (tx, ty) in tiles.index:
    b = tile_bbox(tx, ty)
    tile_crs[(tx, ty)] = utm_epsg((b[0]+b[2])/2, (b[1]+b[3])/2)
print("CRS per tile:", pd.Series(list(tile_crs.values())).value_counts().to_dict())

## Archive baseline for the audit

In [ ]:
(tx0, ty0) = tiles.index[0]
q0 = {"sat:orbit_state": {"eq": ORBIT}} if SOURCE == 's1' else None
_items = list(catalog.search(collections=[COLLECTION],
                             bbox=tile_bbox(tx0, ty0, HALO_DEG),
                             datetime=f"{START_DATE}/{END_DATE}", query=q0).items())
_dates = sorted({pd.Timestamp(i.datetime).tz_convert('UTC').tz_localize(None).date()
                 for i in _items})
EXPECTED_DATES = _dates
print(f"tile {tx0}_{ty0}: {len(_items)} STAC items -> {len(_dates)} distinct acquisition days")
print("first:", _dates[0], "| last:", _dates[-1])
print("\nNOTE: odc groups by solar_day, so the array should have ~this many timesteps.")

## Encoding to uint16

In [ ]:
def encode(arr, band):
    """float/int -> uint16, with 0 reserved for nodata."""
    a = np.asarray(arr)
    bad = ~np.isfinite(a)
    if band == 'SCL':                       # categorical class code, never scaled
        return np.clip(np.where(bad, 0, a), 0, 65535).astype('uint16')
    if ENCODE == 'linear_x10000':
        a = a * 10000.0
    a = np.where(bad, 0, a)
    a = np.clip(a, 0, 65535)
    a = np.where((a < 1) & ~bad, 1, a)      # keep 0 exclusively for nodata
    a = np.where(bad, 0, a)
    return a.astype('uint16')

## Extract one tile

Loads the tile plus halo once per date and cuts every patch from that array. Writes the zarr
store, its attrs, **one JSON for the tile**, and a parquet index.

In [ ]:
HALF = PATCH // 2

def process_tile(tx, ty, tp):
    zip_path = os.path.join(STORE_DIR, f"tile_{tx}_{ty}.zarr.zip")
    meta_p   = os.path.join(STORE_DIR, f"tile_{tx}_{ty}.meta.json")
    if os.path.exists(meta_p) and os.path.exists(zip_path) and not FORCE:
        return 'skipped'
    local = os.path.join(WORK_DIR, f"tile_{tx}_{ty}.zarr")   # local scratch
    if os.path.exists(local):
        shutil.rmtree(local)

    crs  = tile_crs[(tx, ty)]
    bbox = tile_bbox(tx, ty, HALO_DEG)
    q = {"sat:orbit_state": {"eq": ORBIT}} if SOURCE == 's1' else None
    items = list(catalog.search(collections=[COLLECTION], bbox=bbox,
                                datetime=f"{START_DATE}/{END_DATE}", query=q).items())
    if SOURCE == 's2' and MAX_SCENE_CLOUD < 100:
        items = [i for i in items if i.properties.get('eo:cloud_cover', 0) <= MAX_SCENE_CLOUD]
    if not items:
        return 'no_scenes'
    items = [pc.sign(i) for i in items]      # fresh SAS tokens per tile

    ds = odc_load(items, bands=BANDS, bbox=bbox, crs=crs, resolution=RES_M,
                  groupby='solar_day', chunks={'time': 1})
    dates = pd.to_datetime(ds['time'].values)
    n_t, n_b, n_p = len(dates), len(BANDS), len(tp)

    tpp = tp.to_crs(crs)
    xs, ys = ds['x'].values, ds['y'].values
    px = np.array([int(np.argmin(np.abs(xs - v))) for v in tpp.geometry.x.values])
    py = np.array([int(np.argmin(np.abs(ys - v))) for v in tpp.geometry.y.values])
    px = np.clip(px, HALF, len(xs) - HALF)
    py = np.clip(py, HALF, len(ys) - HALF)

    comp = zarr.Blosc(cname='zstd', clevel=5, shuffle=zarr.Blosc.BITSHUFFLE)
    z = zarr.open(local, mode='w', shape=(n_p, n_t, n_b, PATCH, PATCH),
                  chunks=(1, 1, n_b, PATCH, PATCH), dtype='uint16', compressor=comp)

    for ti in range(n_t):
        day = ds.isel(time=ti).compute()                  # one date at a time: flat RAM
        stack = np.stack([encode(day[b].values, b) for b in BANDS], axis=0)
        for pi in range(n_p):
            y0, x0 = py[pi] - HALF, px[pi] - HALF
            z[pi, ti] = stack[:, y0:y0+PATCH, x0:x0+PATCH]
        del day, stack

    date_str = [str(d.date()) for d in dates]
    ids      = [int(v) for v in tp['id'].values]

    # Self-describing store: attrs travel with the data.
    z.attrs.update({
        'run_id': RUN_ID, 'source': SOURCE, 'collection': COLLECTION,
        'dims': ['point','time','band','y','x'], 'bands': BANDS,
        'dtype': 'uint16', 'scale_factor': SCALE_FACTOR, 'nodata': NODATA,
        'encoding': ENCODE, 'crs': crs, 'resolution_m': RES_M, 'patch': PATCH,
        'orbit': ORBIT if SOURCE == 's1' else None,
        'point_ids': ids, 'dates': date_str,
        'decode': 'value * scale_factor where value != 0; SCL categorical, unscaled',
    })
    # Zip the local store and move ONE file to Drive. Writing ~21,000 chunk files
    # directly to a mounted Drive would be unusably slow.
    zarr.consolidate_metadata(local)
    tmp_zip = os.path.join(WORK_DIR, f"tile_{tx}_{ty}.zarr")
    shutil.make_archive(tmp_zip, 'zip', local)             # -> <tmp_zip>.zip
    shutil.move(tmp_zip + '.zip', zip_path)                # single file to Drive
    shutil.rmtree(local)                                   # free the scratch disk

    # ---- ONE JSON PER TILE (not per sample) ----
    # Everything shared by the tile sits at the top level; anything that differs per
    # point sits in 'samples', indexed the same way as axis 0 of the array.
    res = RES_M
    samples = []
    for pi, pid in enumerate(ids):
        x_ul = float(xs[px[pi] - HALF]); y_ul = float(ys[py[pi] - HALF])
        samples.append({
            'id': pid,
            'array_index': pi,
            # affine transform of THIS patch, GDAL order (a, b, c, d, e, f)
            'transform': [res, 0.0, x_ul, 0.0, -res, y_ul],
            'bounds_utm': [x_ul, y_ul - res*PATCH, x_ul + res*PATCH, y_ul],
            'center_lonlat': [float(tp.geometry.x.values[pi]),
                              float(tp.geometry.y.values[pi])],
            'crop': str(tp['crop'].values[pi]),
            'district': str(tp['district'].values[pi]),
            'season': str(tp['season'].values[pi]) if 'season' in tp else None,
            'collection_date': str(tp['collection_date'].values[pi])
                               if 'collection_date' in tp else None,
        })

    tile_meta = {
        'run_id': RUN_ID, 'source': SOURCE, 'collection': COLLECTION,
        'orbit': ORBIT if SOURCE == 's1' else None,
        'tile': [int(tx), int(ty)],
        'store': os.path.basename(zip_path),
        'shape': [n_p, n_t, n_b, PATCH, PATCH],
        'dims': ['point', 'time', 'band', 'y', 'x'],
        'bands': BANDS, 'dtype': 'uint16',
        'scale_factor': SCALE_FACTOR, 'nodata': NODATA, 'encoding': ENCODE,
        'decode': 'value * scale_factor where value != 0; SCL categorical, unscaled',
        'crs': crs, 'resolution_m': res, 'patch': PATCH,
        'n_points': n_p, 'n_timesteps': n_t, 'dates': date_str,
        'point_ids': ids,
        'samples': samples,
    }
    with open(meta_p, 'w') as f:
        json.dump(tile_meta, f, indent=1)

    pd.DataFrame({'id': ids, 'array_index': np.arange(n_p),
                  'store': os.path.basename(zip_path), 'n_timesteps': n_t,
                  'lon': tp.geometry.x.values, 'lat': tp.geometry.y.values,
                  'crop': tp['crop'].values, 'district': tp['district'].values,
                  'crs': crs, 'tile_tx': tx, 'tile_ty': ty}
                 ).to_parquet(os.path.join(STORE_DIR, f"tile_{tx}_{ty}.points.parquet"),
                               index=False)
    del ds
    return 'done'


def process_retry(tx, ty, tp):
    for a in range(1, MAX_RETRIES + 1):
        try:
            return process_tile(tx, ty, tp)
        except Exception as e:
            if a == MAX_RETRIES:
                print(f"  tile {tx}_{ty} FAILED: {type(e).__name__}: {str(e)[:90]}")
                return 'error'
            time.sleep(2 ** a)
    return 'error' 

## Run

In [ ]:
def dirsize(p):
    return sum(os.path.getsize(os.path.join(d, f))
               for d, _, fs in os.walk(p) for f in fs)

if TEST_MODE:
    t0 = time.time()
    r  = process_retry(tx0, ty0, pts)
    dt = time.time() - t0
    store0 = os.path.join(STORE_DIR, f"tile_{tx0}_{ty0}.zarr.zip")
    print("result:", r, f"| {dt/60:.1f} min")
    if r == 'done':
        mb = os.path.getsize(store0) / 1e6
        m  = json.load(open(os.path.join(STORE_DIR, f"tile_{tx0}_{ty0}.meta.json")))
        per_point = mb / m['n_points']
        print(f"{m['n_points']} points x {m['n_dates']} dates -> {mb:,.1f} MB "
              f"({per_point:.1f} MB/point compressed)")
        print(f"\nPROJECTION for the full run ({len(tiles_all)} tiles, "
              f"{len(gpd.read_file(POINTS_GPKG, layer=POINTS_LAYER)):,} points):")
        full_pts = len(gpd.read_file(POINTS_GPKG, layer=POINTS_LAYER))
        print(f"  size : {per_point*full_pts/1e6:,.2f} TB")
        print(f"  time : {dt/m['n_points']*full_pts/3600:,.1f} hours (single-threaded)")
        print("\nCheck this against your free space BEFORE setting TEST_MODE = False.")
else:
    from concurrent.futures import ThreadPoolExecutor, as_completed

    # Each worker stages a whole tile on the Colab VM disk, then holds the store AND
    # its zip briefly while archiving. Peak local use ~= workers x tile_size x 2.
    N_WORKERS = 3          # raise only if the guard below passes with headroom

    # Guard: derive the safe worker count from the tile you actually measured in
    # TEST_MODE, rather than guessing. Set TILE_GB from the test run's printout.
    TILE_GB = None         # <-- set to the test tile's compressed GB, e.g. 2.5
    free_gb = shutil.disk_usage('/content').free / 1e9
    if TILE_GB:
        est_gb = N_WORKERS * TILE_GB * 2 * 1.5      # x2 for store+zip, x1.5 headroom
        safe   = max(1, int(free_gb / (TILE_GB * 2 * 1.5)))
        print(f"local free {free_gb:,.0f} GB | estimated peak {est_gb:,.0f} GB "
              f"| safe workers <= {safe}")
        assert est_gb < free_gb, (
            f"N_WORKERS={N_WORKERS} would need ~{est_gb:,.0f} GB of /content but only "
            f"{free_gb:,.0f} GB is free. Drop to {safe}.")
    else:
        print(f"local free {free_gb:,.0f} GB | TILE_GB not set — guard skipped. "
              "Watch the disk for the first few tiles.")
    print("NOTE: past ~3-4 concurrent tiles you are usually limited by Drive upload "
          "throughput, not by MPC reads, so more workers stop helping.")

    failed, t0 = [], time.time()
    stats = {'done':0,'skipped':0,'no_scenes':0,'error':0}
    tl = list(tiles.items())
    def work(item):
        (a, b), cnt = item
        return a, b, int(cnt), process_retry(a, b, pts[(pts['tx']==a)&(pts['ty']==b)])
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futs = [ex.submit(work, it) for it in tl]
        for i, fut in enumerate(as_completed(futs), 1):
            a, b, cnt, r = fut.result(); stats[r] += 1
            if r in ('error','no_scenes'):
                failed.append({'tx':int(a),'ty':int(b),'reason':r,'points':cnt})
            if i % 5 == 0 or i == len(tl):
                el = time.time() - t0
                fg = shutil.disk_usage('/content').free / 1e9
                print(f"[{i}/{len(tl)}] {stats} | {el/60:.1f} min, "
                      f"~{(len(tl)-i)*el/max(i,1)/60:.0f} min left | "
                      f"local free {fg:,.0f} GB")
                if fg < 10:
                    print("  *** LOW DISK on /content — lower N_WORKERS and restart. ***")

## TIMESTEP AUDIT

Checks: every archive acquisition present in the array; every sample has the full timestep list; no timestep is entirely nodata for a sample.

In [ ]:
stores = sorted(glob.glob(os.path.join(STORE_DIR, "tile_*.zarr.zip")))
assert stores, "no store written — check the run cell"
s0 = stores[0]
zs = zarr.ZipStore(s0, mode='r')          # zarr reads straight from the zip
z  = zarr.open(zs, mode='r')
A  = z.attrs.asdict()
n_p, n_t, n_b = z.shape[0], z.shape[1], z.shape[2]

print("array shape :", z.shape, "| chunks:", z.chunks, "|", z.dtype)
print("bands       :", A['bands'])
print("timesteps   :", n_t)
print("archive said :", len(EXPECTED_DATES), "acquisition days")

missing = sorted(set(str(d) for d in EXPECTED_DATES) - set(A['dates']))
extra   = sorted(set(A['dates']) - set(str(d) for d in EXPECTED_DATES))
print(f"\ndates in archive but NOT in array: {len(missing)}", missing[:5])
print(f"dates in array but not in archive : {len(extra)}", extra[:5])
if not missing:
    print("=> every acquisition the archive reported is present.")
else:
    print("=> *** DATES WERE DROPPED. Do not start the full run. ***")

In [ ]:
# Per-sample, per-timestep validity: fraction of non-nodata pixels.
valid = np.zeros((n_p, n_t), dtype='float32')
for pi in range(n_p):
    for ti in range(n_t):
        valid[pi, ti] = float((z[pi, ti] != A['nodata']).mean())

empty_cells = (valid == 0)
print(f"sample-timestep cells      : {valid.size:,}")
print(f"entirely nodata            : {int(empty_cells.sum()):,} "
      f"({empty_cells.mean():.1%})")
print(f"timesteps empty for ALL    : {int(empty_cells.all(axis=0).sum())} / {n_t}")
print(f"samples with >=1 empty step: {int(empty_cells.any(axis=1).sum())} / {n_p}")
print(f"\nvalid-pixel fraction: min {valid.min():.2f} | "
      f"median {np.median(valid):.2f} | mean {valid.mean():.2f}")

per_sample = pd.DataFrame({
    'id': A['point_ids'],
    'n_timesteps': n_t,
    'steps_with_data': (valid > 0).sum(axis=1),
    'steps_empty': empty_cells.sum(axis=1),
    'mean_valid_frac': valid.mean(axis=1).round(3),
})
print("\nper sample:")
print(per_sample.to_string(index=False))

In [ ]:
# Decode check: do the integers turn back into sensible physical values?
pi = 0
print("sample id:", A['point_ids'][pi])
for ti in [0, n_t//2, n_t-1]:
    patch = z[pi, ti]
    print(f"\n timestep {ti} ({A['dates'][ti]}): "
          f"valid {float((patch != 0).mean()):.1%}")
    for bi, b in enumerate(A['bands']):
        v = patch[bi][patch[bi] != 0]
        if v.size == 0:
            print(f"    {b:>4}: all nodata"); continue
        if b == 'SCL':
            print(f"    {b:>4}: classes {sorted(set(v.tolist()))[:6]}")
        else:
            d = v * A['scale_factor']
            print(f"    {b:>4}: stored {int(v.min()):>6}-{int(v.max()):<6}"
                  f" -> {d.min():.4f}-{d.max():.4f}")
print("\nExpected: S2 reflectance roughly 0-1 | S1 linear gamma0 roughly 0.001-2.")

In [ ]:
# Per-TILE JSON: one file per tile, holding tile-level fields plus a 'samples'
# list whose order matches axis 0 of the array.
mf = sorted(glob.glob(os.path.join(STORE_DIR, "tile_*.meta.json")))
print(f"per-tile JSON files: {len(mf)} (stores: {len(stores)})")

M = json.load(open(s0.replace(".zarr.zip", ".meta.json")))
print("\ntile-level keys:", [k for k in M if k != 'samples'])
print(f"samples listed  : {len(M['samples'])} (points in array: {n_p})")

print("\nexample entry from 'samples':")
print(json.dumps(M['samples'][0], indent=1))

ok = (M['n_timesteps'] == n_t and M['dates'] == A['dates']
      and M['bands'] == A['bands'] and len(M['samples']) == n_p
      and [s['array_index'] for s in M['samples']] == list(range(n_p)))
print("\nJSON agrees with the array:", ok)
assert len(mf) == len(stores), "a tile JSON is missing"
assert ok, "tile JSON does not match the array — do not start the full run"

### Read a patch back

Metadata is one JSON per tile — find the point's tile, then its `array_index`.

```python
import zarr, json, glob, numpy as np

PID = 9570
for mp in glob.glob(f"{STORE_DIR}/tile_*.meta.json"):
    M = json.load(open(mp))
    if PID in M['point_ids']:
        break

s  = next(x for x in M['samples'] if x['id'] == PID)
zs = zarr.ZipStore(f"{STORE_DIR}/{M['store']}", mode='r')
z  = zarr.open(zs, mode='r')

patch = z[s['array_index'], 0]                      # (band, 128, 128) uint16
ref   = np.where(patch == M['nodata'], np.nan, patch * M['scale_factor'])
zs.close()
```

In [ ]:
qa = {
    'generated': pd.Timestamp.now().isoformat(),
    'mode': 'TEST' if TEST_MODE else 'FULL',
    'run_id': RUN_ID, 'source': SOURCE, 'collection': COLLECTION,
    'orbit': ORBIT if SOURCE == 's1' else None,
    'start': START_DATE, 'end': END_DATE,
    'patch': PATCH, 'bands': BANDS, 'dtype': 'uint16',
    'scale_factor': SCALE_FACTOR, 'nodata': NODATA, 'encoding': ENCODE,
    'points': int(n_p), 'timesteps': int(n_t),
    'archive_dates': len(EXPECTED_DATES), 'dates_missing_from_array': missing,
    'empty_cell_fraction': float(empty_cells.mean()),
    'samples_with_empty_step': int(empty_cells.any(axis=1).sum()),
    'store_bytes': int(os.path.getsize(s0)),
}
with open(os.path.join(DIR_QA, f'05_patches_{RUN_ID}_qa.json'), 'w') as f:
    json.dump(qa, f, indent=2, default=str)
print("QA saved.")

---
### Before `TEST_MODE = False`
- `dates in archive but NOT in array` must be 0.
- Check the size projection against free space.
- A few empty cells (swath edge) are normal; whole-timestep-empty is not.

### Colab + Drive notes
- Only 3 files per tile reach Drive (`.zarr.zip`, `.meta.json`, `.points.parquet`); chunk files stay on the VM.
- Local disk is the constraint during the run: `N_WORKERS x tile_size` must fit `/content`.
- Checkpointed on Drive file presence — reconnect and re-run to resume across sessions.

### Still to build
AlphaEarth and Tessera are not on Planetary Computer and need their own notebooks (both annual, 1-2 timesteps).